In [1]:
import torch 
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [2]:
!pip install -q transformers datasets peft accelerate trl bitsandbytes

In [3]:
from transformers import AutoTokenizer,AutoModelForCausalLM
model_name="Qwen/Qwen2.5-0.5B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
print("Model loaded successfully!")
print("Parameters:", f"{model.num_parameters():,}")
print("Device:", model.device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Parameters: 494,032,768
Device: cuda:0


In [4]:

messages = [{"role": "user","content": "Explain artificial intelligence in simple words."}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model.device)

print("Input shape:", inputs["input_ids"].shape)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("\nModel response:\n", response)

Input shape: torch.Size([1, 37])

Model response:
 Artificial Intelligence (AI) is a branch of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence, such as learning, reasoning, perception, and decision-making. In simpler terms, AI involves designing software or hardware systems that can think like humans and mimic the cognitive functions of the brain.

AI systems can be programmed to recognize patterns and make predictions based on data they have been trained on. They can also learn from experience and improve their performance over time through feedback and adaptation. AI applications range from simple tasks like image recognition and speech recognition to complex decision-making processes in


In [5]:
total_params=sum(
    p.numel()
    for p in model.parameters()
)
trainable_params=sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Non-trainable parameters:", f"{total_params - trainable_params:,}")

Total parameters: 494,032,768
Trainable parameters: 494,032,768
Non-trainable parameters: 0


In [6]:
for name, param in model.named_parameters():
    print("Parameter name:", name)
    print("Shape:", param.shape)
    print("Trainable:", param.requires_grad)

Parameter name: model.embed_tokens.weight
Shape: torch.Size([151936, 896])
Trainable: True
Parameter name: model.layers.0.self_attn.q_proj.weight
Shape: torch.Size([896, 896])
Trainable: True
Parameter name: model.layers.0.self_attn.q_proj.bias
Shape: torch.Size([896])
Trainable: True
Parameter name: model.layers.0.self_attn.k_proj.weight
Shape: torch.Size([128, 896])
Trainable: True
Parameter name: model.layers.0.self_attn.k_proj.bias
Shape: torch.Size([128])
Trainable: True
Parameter name: model.layers.0.self_attn.v_proj.weight
Shape: torch.Size([128, 896])
Trainable: True
Parameter name: model.layers.0.self_attn.v_proj.bias
Shape: torch.Size([128])
Trainable: True
Parameter name: model.layers.0.self_attn.o_proj.weight
Shape: torch.Size([896, 896])
Trainable: True
Parameter name: model.layers.0.mlp.gate_proj.weight
Shape: torch.Size([4864, 896])
Trainable: True
Parameter name: model.layers.0.mlp.up_proj.weight
Shape: torch.Size([4864, 896])
Trainable: True
Parameter name: model.layer

In [7]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [8]:
layer = model.model.layers[0]
print(layer)

Qwen2DecoderLayer(
  (self_attn): Qwen2Attention(
    (q_proj): Linear(in_features=896, out_features=896, bias=True)
    (k_proj): Linear(in_features=896, out_features=128, bias=True)
    (v_proj): Linear(in_features=896, out_features=128, bias=True)
    (o_proj): Linear(in_features=896, out_features=896, bias=False)
  )
  (mlp): Qwen2MLP(
    (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
    (up_proj): Linear(in_features=896, out_features=4864, bias=False)
    (down_proj): Linear(in_features=4864, out_features=896, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
  (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
)


In [9]:
layer_params = sum(
    p.numel() for p in layer.parameters()
)

print("Parameters in first layer:", layer_params)

Parameters in first layer: 14912384


In [10]:
total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total:", total_params)
print("Trainable:", trainable_params)
print("Trainable %:", trainable_params / total_params * 100)

Total: 494032768
Trainable: 494032768
Trainable %: 100.0


In [11]:
from peft import LoraConfig,get_peft_model,TaskType

In [12]:
lora_config=LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj"],
    task_type=TaskType.CAUSAL_LM
)

In [13]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [14]:
model=get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [15]:
from datasets import Dataset
data = {
    "messages": [
        [
            {"role": "user", "content": "What is a Java variable?"},
            {"role": "assistant", "content": "A variable is a named storage location used to hold data."}
        ],
        [
            {"role": "user", "content": "What is a Java loop?"},
            {"role": "assistant", "content": "A loop repeatedly executes a block of code."}
        ]
    ]
}
dataset=Dataset.from_dict(data)
print(dataset)

Dataset({
    features: ['messages'],
    num_rows: 2
})


In [16]:
print(dataset[0])

{'messages': [{'content': 'What is a Java variable?', 'role': 'user'}, {'content': 'A variable is a named storage location used to hold data.', 'role': 'assistant'}]}


In [17]:
formatted_text=tokenizer.apply_chat_template(
    dataset[0]["messages"],
    tokenize=False,
    add_generation_prompt=False
)
print(formatted_text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is a Java variable?<|im_end|>
<|im_start|>assistant
A variable is a named storage location used to hold data.<|im_end|>



In [18]:
tokens=tokenizer(
    formatted_text,
    return_tensors="pt"
)
print(tokens)

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   3838,    374,    264,
           7943,   3890,     30, 151645,    198, 151644,  77091,    198,     32,
           3890,    374,    264,   6941,   5819,   3728,   1483,    311,   3331,
            821,     13, 151645,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1]])}


In [19]:
print("Input IDs:")
print(tokens["input_ids"])
print("\nInput shape:")
print(tokens["input_ids"].shape)

Input IDs:
tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   3838,    374,    264,
           7943,   3890,     30, 151645,    198, 151644,  77091,    198,     32,
           3890,    374,    264,   6941,   5819,   3728,   1483,    311,   3331,
            821,     13, 151645,    198]])

Input shape:
torch.Size([1, 49])


In [20]:
decoded_text=tokenizer.decode(
    tokens["input_ids"][0]
)
print(decoded_text)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is a Java variable?<|im_end|>
<|im_start|>assistant
A variable is a named storage location used to hold data.<|im_end|>



In [21]:
def preprocess(example):
    text=tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    tokenized=tokenizer(
        text,
        truncation=True,
        max_length=512
    )
    tokenized["labels"]=tokenized["input_ids"].copy()
    return tokenized
tokenized_dataset=dataset.map(preprocess)
print(tokenized_dataset)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})


In [22]:
from transformers import TrainingArguments,Trainer


In [23]:
training_args=TrainingArguments(
    output_dir="./qwen-lora-output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim="adamw_torch",
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)

In [24]:
print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_star

In [25]:
from transformers import DataCollatorForSeq2Seq
data_collator=DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=100,
    return_tensors="pt"
)

In [26]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [27]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=3, training_loss=4.986008961995442, metrics={'train_runtime': 1.7693, 'train_samples_per_second': 3.391, 'train_steps_per_second': 1.696, 'total_flos': 632286014976.0, 'train_loss': 4.986008961995442, 'epoch': 3.0})

In [28]:
model.save_pretrained("./qwen-lora-adapter")
tokenizer.save_pretrained("./qwen-lora-adapter")

('./qwen-lora-adapter/tokenizer_config.json',
 './qwen-lora-adapter/chat_template.jinja',
 './qwen-lora-adapter/tokenizer.json')

In [29]:
from peft import PeftModel
base_model=AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    "./qwen-lora-adapter"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [30]:
messages = [
    {"role": "user", "content": "What is a Java variable?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

inputs = {
    key: value.to(fine_tuned_model.device)
    for key, value in inputs.items()
}

outputs = fine_tuned_model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

A Java variable is an identifier that holds a value of any type in the Java programming language. It's used to store data and can be assigned a new value at runtime.

Here’s a basic example:

```java
int age = 25;
String name = "John";
double salary = 5000.78;
```

In this example:
- `age` is an integer.
- `name` is a string.
- `salary` is a double.

Variables


In [31]:
def generate_response(model, question):
    messages = [
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

In [32]:
question = "What is a Java variable?"

print("ORIGINAL MODEL:")
print(generate_response(base_model, question))

print("\nFINE-TUNED MODEL:")
print(generate_response(fine_tuned_model, question))

ORIGINAL MODEL:
A Java variable is an identifier that holds a value of any type in the Java programming language. It's used to store data and can be assigned a new value at runtime.

Here’s a basic example:

```java
int age = 25;
String name = "John";
double salary = 5000.78;
```

In this example:
- `age` is an integer.
- `name` is a string.
- `salary` is a double.

Variables

FINE-TUNED MODEL:
A Java variable is an identifier that holds a value of any type in the Java programming language. It's used to store data and can be assigned a new value at runtime.

Here’s a basic example:

```java
int age = 25;
String name = "John";
double salary = 5000.78;
```

In this example:
- `age` is an integer.
- `name` is a string.
- `salary` is a double.

Variables


In [33]:
for name, param in fine_tuned_model.named_parameters():
    if "lora_" in name:
        print(name, param.abs().mean().item())

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight 0.0168149396777153
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight 0.0001968106662388891
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight 0.016711827367544174
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight 0.00019999875803478062
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight 0.016804397106170654
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight 0.0001999104133574292
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight 0.016706913709640503
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight 0.0001999976229853928
base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight 0.016693172976374626
base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight 0.0001999333471758291
base_model.model.model.layers.2.self_attn.v_proj.lora_A.default.weight 0.016

In [34]:
from datasets import Dataset

eval_data = {
    "messages": [
        [
            {"role": "user", "content": "What is inheritance in Java?"},
            {"role": "assistant", "content": "Inheritance is an OOP concept in which one class acquires the properties and methods of another class."}
        ],
        [
            {"role": "user", "content": "What is a Java constructor?"},
            {"role": "assistant", "content": "A constructor is a special method used to initialize objects when they are created."}
        ]
    ]
}

eval_dataset = Dataset.from_dict(eval_data)

print(eval_dataset)

Dataset({
    features: ['messages'],
    num_rows: 2
})


In [35]:
tokenized_eval_dataset = eval_dataset.map(preprocess)

print(tokenized_eval_dataset)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['messages', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})


In [36]:
eval_results = trainer.evaluate(
    eval_dataset=tokenized_eval_dataset
)

print(eval_results)

Training Loss,Validation Loss,Step
No log,4.567905,3


{'eval_loss': 4.567904949188232}


In [37]:
import math

eval_loss = eval_results["eval_loss"]

perplexity = math.exp(eval_loss)

print("Evaluation Loss:", eval_loss)
print("Perplexity:", perplexity)

Evaluation Loss: 4.567904949188232
Perplexity: 96.34205668693942


In [38]:
!pip install -q evaluate rouge_score sacrebleu

In [39]:
references = [
    "A variable is used to store data in a program.",
    "A loop repeatedly executes a block of code."
]

predictions = [
    "A variable stores information in a program.",
    "A loop executes a set of instructions repeatedly."
]

In [40]:
import evaluate

bleu = evaluate.load("sacrebleu")

bleu_result = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

print(bleu_result)

{'score': 21.437389185227836, 'counts': [13, 6, 2, 1], 'totals': [17, 15, 13, 11], 'precisions': [76.47058823529412, 40.0, 15.384615384615385, 9.090909090909092], 'bp': 0.8382234324229999, 'sys_len': 17, 'ref_len': 20}


In [41]:
rouge = evaluate.load("rouge")

rouge_result = rouge.compute(
    predictions=predictions,
    references=references
)

print(rouge_result)

{'rouge1': np.float64(0.6691176470588235), 'rouge2': np.float64(0.34285714285714286), 'rougeL': np.float64(0.6066176470588235), 'rougeLsum': np.float64(0.6066176470588235)}


In [42]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


In [43]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

In [44]:
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [45]:
!pip install -U "bitsandbytes>=0.46.1"

In [46]:
import bitsandbytes as bnb

print("bitsandbytes version:", bnb.__version__)

bitsandbytes version: 0.50.2


In [47]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [48]:
print("Loaded in 4-bit:", model.is_loaded_in_4bit)
print("Memory footprint:", round(model.get_memory_footprint() / 1024**2, 2), "MB")

Loaded in 4-bit: True
Memory footprint: 430.42 MB


In [49]:
model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

In [50]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [51]:
print("4-bit model:", model.base_model.model.is_loaded_in_4bit)

print("\nTrainable parameters:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

4-bit model: True

Trainable parameters:
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([8, 896])
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([896, 8])
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.Size([8, 896])
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.Size([128, 8])
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight torch.Size([8, 896])
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight torch.Size([896, 8])
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight torch.Size([8, 896])
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight torch.Size([128, 8])
base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight torch.Size([8, 896])
base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight torch.Size([896, 8])
base_model.model.model.layers.2.self_at

In [52]:
print("4-bit:", model.base_model.model.is_loaded_in_4bit)

trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total = sum(p.numel() for p in model.parameters())

print("Trainable parameters:", trainable)
print("Total parameters:", total)
print("Trainable percentage:", round(100 * trainable / total, 4), "%")

4-bit: True
Trainable parameters: 540672
Total parameters: 315660160
Trainable percentage: 0.1713 %


In [53]:
!pip install unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 138.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 38.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 70.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 104.6 MB/s  0:00:00
  Attempting uninstall: trl━╺━━━━━━━━━━━━━━━━━━━  5/10 [xformers]]
    Found existing installation: trl 1.17.1━━━━━━━━━━━━━━━━━━━  5/10 [xformers]
    Uninstalling trl-1.17.1:╺━━━━━━━━━━━━━━━━━━━  5/10 [xformers]
      Successfully uninstalled trl-1.17.10m━━━━━━━━━━━━━━━━━━━  5/10 [xformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [unsloth]9/10 [unsloth]zoo]


In [54]:
import unsloth
print("Unsloth installed successfully!")

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1666: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth installed successfully!


In [55]:
from unsloth import FastLanguageModel
import torch

In [56]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=512,
    load_in_4bit=True,
    dtype=torch.float16,
)

==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [57]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.10.11 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [58]:
model.print_trainable_parameters()

trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [59]:
!pip install -q \
    "datasets>=3.4.1,<5.0.0,!=4.0.*,!=4.1.0,!=4.4.*,!=4.5.0" \
    "huggingface-hub>=1.5,<2.0" \
    "pillow>=8,<13"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [60]:
import datasets
import huggingface_hub
import transformers
import bitsandbytes
import unsloth
import peft

print("Datasets:", datasets.__version__)
print("Hub:", huggingface_hub.__version__)
print("Transformers:", transformers.__version__)
print("Bitsandbytes:", bitsandbytes.__version__)
print("PEFT:", peft.__version__)
print("Environment imports successful!")

Datasets: 4.8.5
Hub: 1.29.0
Transformers: 5.16.1
Bitsandbytes: 0.50.2
PEFT: 0.20.0
Environment imports successful!


In [61]:
import requests
from datasets import Dataset, DatasetDict
from collections import Counter

In [62]:
base_url = "https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/"

def download_split(split):
    response = requests.get(base_url + f"{split}.json")
    response.raise_for_status()
    return response.json()

train_data = download_split("train")
dev_data = download_split("dev")
test_data = download_split("test")

print("Training examples:", len(train_data))
print("Validation examples:", len(dev_data))
print("Test examples:", len(test_data))

Training examples: 6251
Validation examples: 883
Test examples: 1147


In [63]:
sample = train_data[0]

print("Question:", sample["qa"]["question"])
print("Answer:", sample["qa"]["answer"])
print("Context:", sample["pre_text"][:2])
print("Table:", sample["table"])
print("Reasoning:", sample["qa"]["program"])

Question: what is the the interest expense in 2009?
Answer: 380
Context: ['interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .', 'if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .']
Table: [['', 'october 31 2009', 'november 1 2008'], ['fair value of forward exchange contracts asset ( liability )', '$ 6427', '$ -23158 ( 23158 )'], ['fair value of forward exchange contracts after a 10% ( 10 % ) unfavorable movement in foreign currency exchange rates asset ( liability )', '$ 20132', '$ -9457 ( 9457 )'], ['fair value of forward exchange contracts after a 10% ( 10 % ) favorable movement in foreign currency exchange rates liability', '$ -6781 ( 6781 )', '$ -38294 ( 38294 )']]
Reasoning: divide(100, 100), divide(3.8, #0)


In [64]:
def prepare_records(records):
    cleaned_records = []

    for record in records:
        qa = record["qa"]

        cleaned_records.append({
            "question": str(qa["question"]),
            "answer": str(qa["answer"]),
            "context": "\n".join(
                record.get("pre_text", []) +
                record.get("post_text", [])
            ),
            "table": str(record.get("table", [])),
            "program": str(qa.get("program", "")),
            "explanation": str(qa.get("explanation", ""))
        })

    return cleaned_records

In [65]:
dataset = DatasetDict({
    "train": Dataset.from_list(prepare_records(train_data)),
    "validation": Dataset.from_list(prepare_records(dev_data)),
    "test": Dataset.from_list(prepare_records(test_data))
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'context', 'table', 'program', 'explanation'],
        num_rows: 6251
    })
    validation: Dataset({
        features: ['question', 'answer', 'context', 'table', 'program', 'explanation'],
        num_rows: 883
    })
    test: Dataset({
        features: ['question', 'answer', 'context', 'table', 'program', 'explanation'],
        num_rows: 1147
    })
})


In [66]:
print("Answer types:")
print(Counter(
    type(answer).__name__
    for answer in dataset["train"]["answer"]
))

print("\nEmpty answers:")
print(sum(
    not answer.strip()
    for answer in dataset["train"]["answer"]
))

print("\nExamples with reasoning programs:")
print(sum(
    bool(program.strip())
    for program in dataset["train"]["program"]
))

Answer types:
Counter({'str': 6251})

Empty answers:
48

Examples with reasoning programs:
6251


In [67]:
sample = dataset["train"][0]

print("QUESTION:\n", sample["question"])
print("\nANSWER:\n", sample["answer"])
print("\nCONTEXT:\n", sample["context"][:1000])
print("\nTABLE:\n", sample["table"][:1000])
print("\nREASONING PROGRAM:\n", sample["program"])

QUESTION:
 what is the the interest expense in 2009?

ANSWER:
 380

CONTEXT:
 interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .
if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .
foreign currency exposure as more fully described in note 2i .
in the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .
dollar-based exposures by entering into forward foreign currency exchange contracts .
the terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .
currently , our largest foreign currency exposure is the euro , primarily because our european operations have the highest proportion of our local currency denominated expenses .
relative to foreign currency exposures existing at october 31 , 2

In [68]:
SYSTEM_PROMPT = """You are a financial question-answering assistant.
Answer questions using the provided financial context and table.
Give a concise answer. Do not invent information."""

In [69]:
def format_example(example):
    user_message = f"""Financial Context:
{example['context']}

Financial Table:
{example['table']}

Question:
{example['question']}"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": example["answer"]}
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

In [70]:
formatted_sample = format_example(dataset["train"][0])
print(formatted_sample["text"])

<|im_start|>system
You are a financial question-answering assistant.
Answer questions using the provided financial context and table.
Give a concise answer. Do not invent information.<|im_end|>
<|im_start|>user
Financial Context:
interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .
if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .
foreign currency exposure as more fully described in note 2i .
in the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .
dollar-based exposures by entering into forward foreign currency exchange contracts .
the terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .
currently , our largest foreign currency exposure is the euro , primarily because our europe

In [71]:
formatted_dataset = DatasetDict({
    split: dataset[split].map(
        format_example,
        remove_columns=dataset[split].column_names
    )
    for split in ["train", "validation", "test"]
})

print(formatted_dataset)

Map:   0%|          | 0/6251 [00:00<?, ? examples/s]

Map:   0%|          | 0/883 [00:00<?, ? examples/s]

Map:   0%|          | 0/1147 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 6251
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 883
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1147
    })
})


In [72]:
def get_token_length(example):
    tokens = tokenizer(
        example["text"],
        truncation=False
    )

    return {
        "token_length": len(tokens["input_ids"])
    }

length_dataset = formatted_dataset["train"].map(
    get_token_length
)

lengths = length_dataset["token_length"]

print("Total examples:", len(lengths))
print("Minimum tokens:", min(lengths))
print("Maximum tokens:", max(lengths))
print("Average tokens:", sum(lengths) / len(lengths))

Map:   0%|          | 0/6251 [00:00<?, ? examples/s]

Total examples: 6251
Minimum tokens: 229
Maximum tokens: 3315
Average tokens: 1084.9979203327468


In [73]:
import numpy as np

for percentile in [50, 75, 90, 95, 99]:
    print(
        f"{percentile}th percentile:",
        int(np.percentile(lengths, percentile))
    )

50th percentile: 1052
75th percentile: 1272
90th percentile: 1499
95th percentile: 1637
99th percentile: 2271


In [74]:
sequence_limit = 2048

over_limit = sum(
    length > sequence_limit
    for length in lengths
)

print("Sequence limit:", sequence_limit)
print("Examples exceeding limit:", over_limit)
print("Examples within limit:", len(lengths) - over_limit)
print("Percentage exceeding limit:", round(over_limit / len(lengths) * 100, 2), "%")

Sequence limit: 2048
Examples exceeding limit: 94
Examples within limit: 6157
Percentage exceeding limit: 1.5 %


In [75]:
max_seq_length = 2048
print("Selected maximum sequence length:", max_seq_length)

Selected maximum sequence length: 2048


In [76]:
from unsloth import FastLanguageModel
model.train()
print("Model training mode:", model.training)
model.print_trainable_parameters()

Model training mode: True
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [77]:
from trl import SFTTrainer, SFTConfig
training_args = SFTConfig(
    output_dir="finqa-qwen-qlora",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2,
    warmup_ratio=0.03,
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_length=max_seq_length,
    packing=False,
    dataset_text_field="text",
    completion_only_loss=True,
    report_to="none",
    seed=3407
)

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


In [78]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    args=training_args
)

print("Trainer created successfully!")

Unsloth: Padded batches + per-block compile chosen over padding-free (host bound (padded micro batch ~89 ms of GPU work)). Set UNSLOTH_PADDING_CHOICE=padding_free to override.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/6251 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/883 [00:00<?, ? examples/s]

Trainer created successfully!


In [79]:
training_result = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,251 | Num Epochs = 2 | Total steps = 1,564
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,399,104 of 498,431,872 (0.88% trained)


Step,Training Loss,Validation Loss
100,1.804411,1.796968
200,1.607735,1.716206
300,1.545042,1.664069
400,1.541473,1.630322
500,1.542675,1.607427
600,1.401514,1.578874
700,1.417846,1.567043
800,1.217813,1.574163
900,1.219519,1.569124
1000,1.227570,1.565201


Unsloth: Not an error, but `num_items_in_batch` could not be counted for Qwen2ForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in finqa-qwen-qlora/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in finqa-qwen-qlora/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in finqa-qwen-qlora/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in finqa-qwen-qlora/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in finqa-qwen-qlora/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in finqa-qwen-qlora/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in finqa-qwen-qlora/checkpoint-700

In [80]:
model.save_pretrained("finqa-qwen-lora")
tokenizer.save_pretrained("finqa-qwen-lora")
print("Fine-tuned adapter and tokenizer saved!")

Unsloth: Restored added_tokens_decoder metadata in finqa-qwen-lora/tokenizer_config.json.


Fine-tuned adapter and tokenizer saved!


In [81]:
import os
print(os.listdir("finqa-qwen-lora"))

['tokenizer_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'README.md', 'adapter_config.json']


In [82]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=torch.float16,
)
print("Original base model loaded successfully!")

==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Original base model loaded successfully!


In [83]:
from peft import PeftModel
finetuned_model = PeftModel.from_pretrained(
    base_model,
    "finqa-qwen-lora"
)
print("Fine-tuned model loaded successfully!")

Fine-tuned model loaded successfully!


In [84]:
def generate_answer(model, tokenizer, question, context, table):

    user_message = f"""Financial Context:
{context}

Financial Table:
{table}

Question:
{question}"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_tokens = outputs[0][inputs.shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [85]:
sample = dataset["test"][0]
question = sample["question"]
context = sample["context"]
table = sample["table"]
actual_answer = sample["answer"]

original_answer = generate_answer(
    base_model,
    base_tokenizer,
    question,
    context,
    table
)
with finetuned_model.disable_adapter():
    original_answer = generate_answer(
        finetuned_model,
        tokenizer,
        question,
        context,
        table
    )
finetuned_answer = generate_answer(
    finetuned_model,
    tokenizer,
    question,
    context,
    table
)

print("=" * 60)
print("QUESTION:")
print(question)

print("\nACTUAL ANSWER:")
print(actual_answer)

print("\nORIGINAL MODEL:")
print(original_answer)

print("\nFINE-TUNED MODEL:")
print(finetuned_answer)

print("=" * 60)

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
what is the net change in net revenue during 2015 for entergy corporation?

ACTUAL ANSWER:
94

ORIGINAL MODEL:
The net change in net revenue during 2015 for Entergy Corporation was:

$ -14 (14) million

This can be calculated by subtracting the amount of expenses (charges, taxes, etc.) from the amount of revenues (net income).

FINE-TUNED MODEL:
2022 decrease of 187.0


In [86]:
raw_sample = test_data[0]

print("QUESTION:")
print(raw_sample["qa"]["question"])

print("\nEXPECTED ANSWER:")
print(raw_sample["qa"]["answer"])

print("\nEXECUTED ANSWER:")
print(raw_sample["qa"].get("exe_ans"))

print("\nREASONING PROGRAM:")
print(raw_sample["qa"].get("program"))

print("\nEXPLANATION:")
print(raw_sample["qa"].get("explanation"))

print("\nFINANCIAL TABLE:")
print(raw_sample["table"])

QUESTION:
what is the net change in net revenue during 2015 for entergy corporation?

EXPECTED ANSWER:
94

EXECUTED ANSWER:
94.0

REASONING PROGRAM:
subtract(5829, 5735)

EXPLANATION:


FINANCIAL TABLE:
[['', 'amount ( in millions )'], ['2014 net revenue', '$ 5735'], ['retail electric price', '187'], ['volume/weather', '95'], ['waterford 3 replacement steam generator provision', '-32 ( 32 )'], ['miso deferral', '-35 ( 35 )'], ['louisiana business combination customer credits', '-107 ( 107 )'], ['other', '-14 ( 14 )'], ['2015 net revenue', '$ 5829']]


In [87]:
print(formatted_dataset["train"][0]["text"])

<|im_start|>system
You are a financial question-answering assistant.
Answer questions using the provided financial context and table.
Give a concise answer. Do not invent information.<|im_end|>
<|im_start|>user
Financial Context:
interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .
if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .
foreign currency exposure as more fully described in note 2i .
in the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .
dollar-based exposures by entering into forward foreign currency exchange contracts .
the terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .
currently , our largest foreign currency exposure is the euro , primarily because our europe

In [88]:
def generate_answer(model, tokenizer, example):

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"""Financial Context:
{example['context']}

Financial Table:
{example['table']}

Question:
{example['question']}"""
        }
    ]

    # Preserve the end of long prompts
    old_truncation_side = tokenizer.truncation_side
    tokenizer.truncation_side = "left"

    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            truncation=True,
            max_length=2048
        ).to(model.device)
    finally:
        tokenizer.truncation_side = old_truncation_side

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[0, inputs.shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

In [89]:
from tqdm.auto import tqdm

NUM_EXAMPLES = 100

test_samples = dataset["test"].select(
    range(NUM_EXAMPLES)
)

results = []

for i, example in enumerate(tqdm(test_samples)):

    with finetuned_model.disable_adapter():
        base_prediction = generate_answer(
            finetuned_model,
            tokenizer,
            example
        )

    tuned_prediction = generate_answer(
        finetuned_model,
        tokenizer,
        example
    )

    results.append({
        "question": example["question"],
        "actual": example["answer"],
        "base_prediction": base_prediction,
        "finetuned_prediction": tuned_prediction
    })

print("Evaluation completed:", len(results))

  0%|          | 0/100 [00:00<?, ?it/s]

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

Evaluation completed: 100


In [93]:
import re
def normalize_answer(answer):
    return str(answer).strip().lower().replace(",", "")

def extract_number(answer):
    match = re.search(r"-?\d+(?:\.\d+)?", str(answer).replace(",", ""))
    return float(match.group()) if match else None

base_exact = 0
tuned_exact = 0
base_numeric = 0
tuned_numeric = 0

for result in results:
    actual = normalize_answer(result["actual"])

    base = normalize_answer(result["base_prediction"])
    tuned = normalize_answer(result["finetuned_prediction"])

    if base == actual:
        base_exact += 1

    if tuned == actual:
        tuned_exact += 1

    actual_num = extract_number(actual)
    base_num = extract_number(base)
    tuned_num = extract_number(tuned)

    if actual_num is not None and base_num is not None:
        if abs(actual_num - base_num) < 1e-6:
            base_numeric += 1

    if actual_num is not None and tuned_num is not None:
        if abs(actual_num - tuned_num) < 1e-6:
            tuned_numeric += 1

print("BASE MODEL")
print("Exact match:", base_exact / NUM_EXAMPLES * 100, "%")
print("Numeric match:", base_numeric / NUM_EXAMPLES * 100, "%")

print("\nFINE-TUNED MODEL")
print("Exact match:", tuned_exact / NUM_EXAMPLES * 100, "%")
print("Numeric match:", tuned_numeric / NUM_EXAMPLES * 100, "%")

BASE MODEL
Exact match: 0.0 %
Numeric match: 1.0 %

FINE-TUNED MODEL
Exact match: 2.0 %
Numeric match: 0.0 %


In [94]:
for result in results[:15]:
    print("\nQUESTION:", result["question"])
    print("EXPECTED:", result["actual"])
    print("BASE:", result["base_prediction"])
    print("FINE-TUNED:", result["finetuned_prediction"])
    print("-" * 70)


QUESTION: what is the net change in net revenue during 2015 for entergy corporation?
EXPECTED: 94
BASE: The net change in net revenue during 2015 for Entergy Corporation was:

$ -14 (14) million

This can be calculated by subtracting the amount of expenses (charges, taxes, etc.) from the amount of revenues (net income).
FINE-TUNED: 2022 decrease of 187.0
----------------------------------------------------------------------

QUESTION: what percentage of total facilities as measured in square feet are leased?
EXPECTED: 14%
BASE: The percentage of total facilities as measured in square feet that are leased is 60%.
FINE-TUNED: 28%
----------------------------------------------------------------------

QUESTION: what is the percentage change in cash flow hedges in 2011 compare to the 2010?
EXPECTED: 9.9%
BASE: The percentage change in cash flow hedges in 2011 compared to 2010 was -20692%.
FINE-TUNED: -18.1%
----------------------------------------------------------------------

QUESTION: 

In [95]:
import torch
import numpy as np

print("=" * 60)
print("1. TRAINER DATASET INSPECTION")
print("=" * 60)

print("Dataset columns:", trainer.train_dataset.column_names)
print("First training example:")
print(trainer.train_dataset[0])

print("\n" + "=" * 60)
print("2. ACTUAL TRAINING LABEL AUDIT")
print("=" * 60)

# Fetch one real batch from the training dataloader
batch = next(iter(trainer.get_train_dataloader()))

input_ids = batch["input_ids"]
labels = batch["labels"]

print("Input shape:", tuple(input_ids.shape))
print("Label shape:", tuple(labels.shape))

for i in range(min(2, input_ids.shape[0])):
    valid_mask = labels[i] != -100

    supervised_labels = labels[i][valid_mask]
    total_tokens = input_ids[i].numel()
    supervised_tokens = valid_mask.sum().item()

    print(f"\n--- Sample {i + 1} ---")
    print("Total tokens:", total_tokens)
    print("Tokens receiving loss:", supervised_tokens)
    print("Tokens ignored (-100):", total_tokens - supervised_tokens)
    print("Supervised percentage:",
          round(100 * supervised_tokens / total_tokens, 2), "%")

    print("\nDecoded supervised text:")
    print(tokenizer.decode(
        supervised_labels,
        skip_special_tokens=False
    )[:1500])

    print("\nDecoded original input:")
    print(tokenizer.decode(
        input_ids[i],
        skip_special_tokens=False
    )[:1000])


print("\n" + "=" * 60)
print("3. TRAINING DATA TRUNCATION AUDIT")
print("=" * 60)

lengths = []

for text in formatted_dataset["train"]["text"]:
    tokenized = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False
    )
    lengths.append(len(tokenized["input_ids"]))

lengths = np.array(lengths)

print("Total examples:", len(lengths))
print("Maximum length:", lengths.max())
print("Average length:", round(lengths.mean(), 2))
print("Median length:", np.percentile(lengths, 50))
print("Examples exceeding 2048:", (lengths > 2048).sum())
print("Percentage exceeding 2048:",
      round(100 * (lengths > 2048).mean(), 2), "%")

print("\nLength percentiles:")
for p in [75, 90, 95, 99]:
    print(f"P{p}:", round(np.percentile(lengths, p), 2))

print("\n" + "=" * 60)
print("AUDIT COMPLETE")
print("=" * 60)

1. TRAINER DATASET INSPECTION
Dataset columns: ['input_ids', 'attention_mask']
First training example:
{'input_ids': [151644, 8948, 198, 2610, 525, 264, 5896, 3405, 12, 596, 86, 4671, 17847, 624, 16141, 4755, 1667, 279, 3897, 5896, 2266, 323, 1965, 624, 35127, 264, 63594, 4226, 13, 3155, 537, 17023, 1995, 13, 151645, 198, 151644, 872, 198, 56230, 9608, 510, 12724, 4379, 311, 264, 3890, 2734, 4379, 3118, 389, 279, 2326, 22289, 3051, 269, 5519, 220, 17, 13, 15, 20, 4, 320, 220, 17, 13, 15, 20, 1018, 873, 320, 220, 17, 13, 18, 19, 4, 320, 220, 17, 13, 18, 19, 1018, 873, 438, 315, 18491, 5923, 220, 18, 16, 1154, 220, 17, 15, 15, 24, 873, 16448, 333, 3051, 269, 4344, 553, 220, 16, 15, 15, 8037, 3501, 1154, 1039, 9775, 2734, 20284, 1035, 2297, 553, 400, 220, 18, 13, 23, 3526, 16448, 28443, 11413, 14338, 438, 803, 7225, 7481, 304, 5185, 220, 17, 72, 16448, 258, 279, 8388, 311, 59291, 5896, 12239, 12985, 304, 1509, 220, 23, 315, 419, 9775, 1895, 389, 1352, 220, 16, 15, 12646, 1154, 582, 15502,

In [96]:
from datasets import Dataset, DatasetDict

def create_prompt_completion(example):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""Financial Context:
{example['context']}

Financial Table:
{example['table']}

Question:
{example['question']}"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    completion = example["answer"] + tokenizer.eos_token

    return {
        "prompt": prompt,
        "completion": completion
    }


corrected_data = {}

for split in ["train", "validation", "test"]:

    corrected_data[split] = Dataset.from_list([
        create_prompt_completion(example)
        for example in dataset[split]
    ])

corrected_dataset = DatasetDict(corrected_data)

print(corrected_dataset)
print("\nSample prompt:\n")
print(corrected_dataset["train"][0]["prompt"][-700:])

print("\nSample completion:\n")
print(corrected_dataset["train"][0]["completion"])

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 6251
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 883
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 1147
    })
})

Sample prompt:

y exchange rates does not factor in a potential change in sales levels or local currency selling prices. .

Financial Table:
[['', 'october 31 2009', 'november 1 2008'], ['fair value of forward exchange contracts asset ( liability )', '$ 6427', '$ -23158 ( 23158 )'], ['fair value of forward exchange contracts after a 10% ( 10 % ) unfavorable movement in foreign currency exchange rates asset ( liability )', '$ 20132', '$ -9457 ( 9457 )'], ['fair value of forward exchange contracts after a 10% ( 10 % ) favorable movement in foreign currency exchange rates liability', '$ -6781 ( 6781 )', '$ -38294 ( 38294 )']]

Question:
what is the the interest expense in 2009?<|im_end|>


In [97]:
from trl import SFTConfig, SFTTrainer

corrected_args = SFTConfig(
    output_dir="finqa-corrected-audit",

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    num_train_epochs=2,
    warmup_ratio=0.03,

    fp16=True,
    bf16=False,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,

    save_strategy="no",

    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    max_length=2048,
    packing=False,

    completion_only_loss=True,

    report_to="none",
    seed=3407
)

corrected_trainer = SFTTrainer(
    model=trainer.model,
    tokenizer=tokenizer,
    train_dataset=corrected_dataset["train"],
    eval_dataset=corrected_dataset["validation"],
    args=corrected_args
)

print("Corrected audit trainer initialized.")
print("Columns:", corrected_trainer.train_dataset.column_names)

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/6251 [00:00<?, ? examples/s]

Unsloth: Building labels for train dataset (num_proc=2):   0%|          | 0/6251 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/883 [00:00<?, ? examples/s]

Unsloth: Building labels for eval dataset (num_proc=2):   0%|          | 0/883 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Corrected audit trainer initialized.
Columns: ['input_ids', 'labels']


In [98]:
import torch

batch = next(iter(corrected_trainer.get_train_dataloader()))

input_ids = batch["input_ids"]
labels = batch["labels"]

print("Input shape:", input_ids.shape)
print("Label shape:", labels.shape)

for i in range(min(2, input_ids.shape[0])):

    valid_mask = labels[i] != -100

    supervised_tokens = labels[i][valid_mask]
    ignored_tokens = (~valid_mask).sum().item()

    print(f"\n{'=' * 50}")
    print(f"SAMPLE {i + 1}")
    print(f"{'=' * 50}")

    print("Total tokens:", input_ids[i].numel())
    print("Tokens receiving loss:", len(supervised_tokens))
    print("Tokens ignored:", ignored_tokens)

    print("\nSUPERVISED TEXT:")
    print(tokenizer.decode(
        supervised_tokens,
        skip_special_tokens=False
    ))

    print("\nEXPECTED COMPLETION:")
    print(corrected_dataset["train"][i]["completion"])

    print("\nSUPERVISED TOKEN COUNT:",
          len(supervised_tokens))

Input shape: torch.Size([1, 1024])
Label shape: torch.Size([1, 1024])

SAMPLE 1
Total tokens: 1024
Tokens receiving loss: 0
Tokens ignored: 1024

SUPERVISED TEXT:


EXPECTED COMPLETION:
380<|im_end|>

SUPERVISED TOKEN COUNT: 0


In [99]:
print("TRAIN DATASET LABEL AUDIT")
print("=" * 55)

train_ds = corrected_trainer.train_dataset

print("Columns:", train_ds.column_names)
print("Total rows:", len(train_ds))

zero_label_count = 0
nonzero_label_count = 0

for i in range(min(20, len(train_ds))):
    row = train_ds[i]

    ids = row["input_ids"]
    labels = row["labels"]

    supervised = sum(
        1 for token in labels if token != -100
    )

    if supervised == 0:
        zero_label_count += 1
    else:
        nonzero_label_count += 1

    print(
        f"Example {i}: "
        f"input={len(ids)}, "
        f"supervised={supervised}"
    )

    if i < 3:
        print("Decoded labels:")
        print(tokenizer.decode(
            [x for x in labels if x != -100],
            skip_special_tokens=False
        ))
        print("-" * 40)

print("\nSUMMARY")
print("Zero-label examples:", zero_label_count)
print("Nonzero-label examples:", nonzero_label_count)

print("\nFIRST DATASET ROW")
print("Input tokens:", len(train_ds[0]["input_ids"]))
print("Label tokens:", len(train_ds[0]["labels"]))
print("Expected completion:",
      corrected_dataset["train"][0]["completion"])

TRAIN DATASET LABEL AUDIT
Columns: ['input_ids', 'labels']
Total rows: 6251
Example 0: input=512, supervised=0
Decoded labels:

----------------------------------------
Example 1: input=512, supervised=0
Decoded labels:

----------------------------------------
Example 2: input=512, supervised=0
Decoded labels:

----------------------------------------
Example 3: input=512, supervised=0
Example 4: input=512, supervised=0
Example 5: input=512, supervised=0
Example 6: input=512, supervised=0
Example 7: input=512, supervised=0
Example 8: input=512, supervised=0
Example 9: input=512, supervised=0
Example 10: input=512, supervised=0
Example 11: input=512, supervised=0
Example 12: input=512, supervised=0
Example 13: input=512, supervised=0
Example 14: input=512, supervised=0
Example 15: input=512, supervised=0
Example 16: input=512, supervised=0
Example 17: input=512, supervised=0
Example 18: input=512, supervised=0
Example 19: input=512, supervised=0

SUMMARY
Zero-label examples: 20
Nonzero

In [100]:
row = corrected_trainer.train_dataset[0]

input_ids = row["input_ids"]
labels = row["labels"]

print("INPUT LENGTH:", len(input_ids))
print("SUPERVISED TOKENS:", sum(x != -100 for x in labels))

print("\nFIRST 30 INPUT TOKENS:")
print(tokenizer.convert_ids_to_tokens(input_ids[:30]))

print("\nLAST 100 INPUT TOKENS:")
print(tokenizer.convert_ids_to_tokens(input_ids[-100:]))

print("\nEXPECTED COMPLETION:")
print(repr(corrected_dataset["train"][0]["completion"]))

print("\nCOMPLETION TOKEN IDS:")
print(tokenizer(
    corrected_dataset["train"][0]["completion"],
    add_special_tokens=False
)["input_ids"])

print("\nLAST 30 LABEL VALUES:")
print(labels[-30:])

print("\nTRAINER CONFIG:")
print("completion_only_loss:", corrected_trainer.args.completion_only_loss)
print("max_length:", corrected_trainer.args.max_length)

INPUT LENGTH: 512
SUPERVISED TOKENS: 0

FIRST 30 INPUT TOKENS:
['<|im_start|>', 'system', 'Ċ', 'You', 'Ġare', 'Ġa', 'Ġfinancial', 'Ġquestion', '-', 'ans', 'w', 'ering', 'Ġassistant', '.Ċ', 'Answer', 'Ġquestions', 'Ġusing', 'Ġthe', 'Ġprovided', 'Ġfinancial', 'Ġcontext', 'Ġand', 'Ġtable', '.Ċ', 'Give', 'Ġa', 'Ġconcise', 'Ġanswer', '.', 'ĠDo']

LAST 100 INPUT TOKENS:
['while', 'Ġthe', 'Ġcontract', 'Ġor', 'Ġnot', 'ional', 'Ġamounts', 'Ġof', 'Ġderivative', 'Ġfinancial', 'Ġinstruments', 'Ġprovide', 'Ġone', 'Ġmeasure', 'Ġof', 'Ġthe', 'Ġvolume', 'Ġof', 'Ġthese', 'Ġtransactions', 'Ġ,', 'Ġthey', 'Ġdo', 'Ġnot', 'Ġrepresent', 'Ġthe', 'Ġamount', 'Ġof', 'Ġour', 'Ġexposure', 'Ġto', 'Ġcredit', 'Ġrisk', 'Ġ.Ċ', 'the', 'Ġamounts', 'Ġpotentially', 'Ġsubject', 'Ġto', 'Ġcredit', 'Ġrisk', 'Ġ(', 'Ġarising', 'Ġfrom', 'Ġthe', 'Ġpossible', 'Ġinability', 'Ġof', 'Ġcounterpart', 'ies', 'Ġto', 'Ġmeet', 'Ġthe', 'Ġterms', 'Ġof', 'Ġtheir', 'Ġcontracts', 'Ġ)', 'Ġare', 'Ġgenerally', 'Ġlimited', 'Ġto', 'Ġthe', 'Ġamounts',

In [101]:
import inspect

print("Trainer preprocessing configuration")
print("=" * 50)

print("Unsloth version:", __import__("unsloth").__version__)
print("TRL version:", __import__("trl").__version__)

print("\nTrainer tokenizer configuration:")
print("Model max sequence length:",
      getattr(corrected_trainer.model, "max_seq_length", None))

print("Tokenizer model max length:",
      tokenizer.model_max_length)

print("\nProcessed example boundaries:")
row = corrected_trainer.train_dataset[0]

print("First tokens:")
print(tokenizer.decode(row["input_ids"][:20]))

print("\nLast tokens:")
print(tokenizer.decode(row["input_ids"][-20:]))

print("\nOriginal prompt token count:",
      len(tokenizer(
          corrected_dataset["train"][0]["prompt"],
          add_special_tokens=False
      )["input_ids"]))

print("\nCompletion token count:",
      len(tokenizer(
          corrected_dataset["train"][0]["completion"],
          add_special_tokens=False
      )["input_ids"]))

Trainer preprocessing configuration
Unsloth version: 2026.10.11
TRL version: 1.15.0

Trainer tokenizer configuration:
Model max sequence length: 512
Tokenizer model max length: 32768

Processed example boundaries:
First tokens:
<|im_start|>system
You are a financial question-answering assistant.
Answer questions using the provided financial

Last tokens:
 the contracts exceed our obligations to the counterparties .
the following table illustrates the effect that a 1

Original prompt token count: 980

Completion token count: 4


In [102]:
from unsloth import FastLanguageModel
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

fresh_model, fresh_tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=torch.float16,
)

print("Model sequence length:", fresh_model.max_seq_length)
print("Tokenizer max length:", fresh_tokenizer.model_max_length)

==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model sequence length: 2048
Tokenizer max length: 32768


In [103]:
fresh_model = FastLanguageModel.get_peft_model(
    fresh_model,
    r=8,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

fresh_model.print_trainable_parameters()

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [104]:
from trl import SFTConfig, SFTTrainer

audit_args = SFTConfig(
    output_dir="finqa-corrected-audit",

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    num_train_epochs=2,
    warmup_ratio=0.03,

    fp16=True,
    bf16=False,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,

    save_strategy="no",

    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    max_length=2048,
    packing=False,

    completion_only_loss=True,

    report_to="none",
    seed=3407,
)

audit_trainer = SFTTrainer(
    model=fresh_model,
    tokenizer=fresh_tokenizer,
    train_dataset=corrected_dataset["train"],
    eval_dataset=corrected_dataset["validation"],
    args=audit_args,
)

print("Model sequence length:", fresh_model.max_seq_length)
print("Trainer max length:", audit_trainer.args.max_length)

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/6251 [00:00<?, ? examples/s]

Unsloth: Building labels for train dataset (num_proc=2):   0%|          | 0/6251 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/883 [00:00<?, ? examples/s]

Unsloth: Building labels for eval dataset (num_proc=2):   0%|          | 0/883 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Model sequence length: 2048
Trainer max length: None


In [105]:
batch = next(iter(audit_trainer.get_train_dataloader()))

for i in range(min(2, batch["input_ids"].shape[0])):

    labels = batch["labels"][i]
    input_ids = batch["input_ids"][i]

    supervised = labels != -100

    print("\nTotal tokens:", input_ids.numel())
    print("Supervised tokens:", supervised.sum().item())
    print("Ignored tokens:", (~supervised).sum().item())

    print("Supervised text:")
    print(fresh_tokenizer.decode(
        labels[supervised],
        skip_special_tokens=False
    ))

    print("Expected answer:")
    print(corrected_dataset["train"][i]["completion"])


Total tokens: 2252
Supervised tokens: 10
Ignored tokens: 2242
Supervised text:
10.3<|im_end|>9.5%<|im_end|>
Expected answer:
380<|im_end|>


In [106]:
row = audit_trainer.train_dataset[0]

input_ids = row["input_ids"]
labels = row["labels"]

supervised_mask = [
    token != -100 for token in labels
]

supervised_ids = [
    token for token in labels if token != -100
]

expected = corrected_dataset["train"][0]["completion"]

print("INPUT TOKENS:", len(input_ids))
print("SUPERVISED TOKENS:", len(supervised_ids))

print("\nEXPECTED COMPLETION:")
print(repr(expected))

print("\nACTUAL SUPERVISED TEXT:")
print(fresh_tokenizer.decode(
    supervised_ids,
    skip_special_tokens=False
))

print("\nEXPECTED TOKEN IDS:")
print(fresh_tokenizer(
    expected,
    add_special_tokens=False
)["input_ids"])

print("\nACTUAL SUPERVISED TOKEN IDS:")
print(supervised_ids)

print("\nPROMPT MASKING:")
print("Ignored prompt tokens:",
      sum(token == -100 for token in labels))

INPUT TOKENS: 984
SUPERVISED TOKENS: 4

EXPECTED COMPLETION:
'380<|im_end|>'

ACTUAL SUPERVISED TEXT:
380<|im_end|>

EXPECTED TOKEN IDS:
[18, 23, 15, 151645]

ACTUAL SUPERVISED TOKEN IDS:
[18, 23, 15, 151645]

PROMPT MASKING:
Ignored prompt tokens: 980


In [107]:
def audit_completion_preservation(split):
    ds = audit_trainer.train_dataset if split == "train" else audit_trainer.eval_dataset
    source = corrected_dataset[split]

    zero_supervision = 0
    incomplete_answers = 0

    for i in range(len(ds)):
        labels = ds[i]["labels"]
        actual = [token for token in labels if token != -100]

        expected = fresh_tokenizer(
            source[i]["completion"],
            add_special_tokens=False
        )["input_ids"]

        if len(actual) == 0:
            zero_supervision += 1

        if actual != expected:
            incomplete_answers += 1

    print(f"\n{split.upper()} AUDIT")
    print("Total examples:", len(ds))
    print("Zero supervised examples:", zero_supervision)
    print("Missing or altered completions:", incomplete_answers)

    return zero_supervision == 0 and incomplete_answers == 0


train_ok = audit_completion_preservation("train")
validation_ok = audit_completion_preservation("validation")

print("\nREADY TO TRAIN:", train_ok and validation_ok)


TRAIN AUDIT
Total examples: 6251
Zero supervised examples: 93
Missing or altered completions: 94

VALIDATION AUDIT
Total examples: 883
Zero supervised examples: 10
Missing or altered completions: 10

READY TO TRAIN: False


In [108]:
import numpy as np

for split in ["train", "validation", "test"]:

    lengths = []

    for example in corrected_dataset[split]:

        prompt_ids = fresh_tokenizer(
            example["prompt"],
            add_special_tokens=False
        )["input_ids"]

        completion_ids = fresh_tokenizer(
            example["completion"],
            add_special_tokens=False
        )["input_ids"]

        lengths.append(len(prompt_ids) + len(completion_ids))

    lengths = np.array(lengths)

    print(f"\n{split.upper()}")
    print("-" * 40)
    print("Examples:", len(lengths))
    print("Maximum:", lengths.max())
    print("Average:", round(lengths.mean(), 2))
    print("Over 2048:", (lengths > 2048).sum())
    print("Over 3072:", (lengths > 3072).sum())
    print("Over 4096:", (lengths > 4096).sum())


TRAIN
----------------------------------------
Examples: 6251
Maximum: 3314
Average: 1084.0
Over 2048: 94
Over 3072: 14
Over 4096: 0

VALIDATION
----------------------------------------
Examples: 883
Maximum: 2185
Average: 1073.11
Over 2048: 10
Over 3072: 0
Over 4096: 0

TEST
----------------------------------------
Examples: 1147
Maximum: 2563
Average: 1072.21
Over 2048: 18
Over 3072: 0
Over 4096: 0


In [109]:

fresh_model, fresh_tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=torch.float16,
)

fresh_model = FastLanguageModel.get_peft_model(
    fresh_model,
    r=8,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("Model context length:", fresh_model.max_seq_length)
print("Tokenizer:", fresh_tokenizer.name_or_path)
print("LoRA adapters attached successfully.")

==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Model context length: 4096
Tokenizer: unsloth/Qwen2.5-0.5B-Instruct-unsloth-bnb-4bit
LoRA adapters attached successfully.


In [110]:
audit_args = SFTConfig(
    output_dir="finqa-corrected-audit",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=2,
    warmup_ratio=0.03,
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="no",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_length=4096,
    packing=False,
    completion_only_loss=True,
    report_to="none",
    seed=3407,
)

audit_trainer = SFTTrainer(
    model=fresh_model,
    tokenizer=fresh_tokenizer,
    train_dataset=corrected_dataset["train"],
    eval_dataset=corrected_dataset["validation"],
    args=audit_args,
)

print("Audit trainer prepared.")
print("Max sequence length:", fresh_model.max_seq_length)

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/6251 [00:00<?, ? examples/s]

Unsloth: Building labels for train dataset (num_proc=2):   0%|          | 0/6251 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/883 [00:00<?, ? examples/s]

Unsloth: Building labels for eval dataset (num_proc=2):   0%|          | 0/883 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Audit trainer prepared.
Max sequence length: 4096


In [111]:
def audit_completion_preservation(split):
    ds = (
        audit_trainer.train_dataset
        if split == "train"
        else audit_trainer.eval_dataset
    )
    source = corrected_dataset[split]

    zero_supervision = 0
    incomplete_answers = 0

    for i in range(len(ds)):
        labels = ds[i]["labels"]

        actual = [token for token in labels if token != -100]

        expected = fresh_tokenizer(
            source[i]["completion"],
            add_special_tokens=False
        )["input_ids"]

        if len(actual) == 0:
            zero_supervision += 1

        if actual != expected:
            incomplete_answers += 1

    print(f"\n{split.upper()} AUDIT")
    print("Total examples:", len(ds))
    print("Zero supervised:", zero_supervision)
    print("Missing/altered completions:", incomplete_answers)

    return zero_supervision == 0 and incomplete_answers == 0


train_ready = audit_completion_preservation("train")
validation_ready = audit_completion_preservation("validation")

print("\nREADY TO TRAIN:", train_ready and validation_ready)


TRAIN AUDIT
Total examples: 6251
Zero supervised: 0
Missing/altered completions: 0

VALIDATION AUDIT
Total examples: 883
Zero supervised: 0
Missing/altered completions: 0

READY TO TRAIN: True


In [112]:
training_result = audit_trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,251 | Num Epochs = 2 | Total steps = 1,564
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 4,399,104 of 498,431,872 (0.88% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,1.724177,1.668968
200,1.641367,1.583068
300,1.533430,1.548562
400,1.495599,1.513885
500,1.486105,1.503986
600,1.403223,1.504491
700,1.392986,1.466662
800,1.141703,1.442756
900,1.295298,1.421624
1000,1.238956,1.412260


Filter:   0%|          | 0/883 [00:00<?, ? examples/s]